# Ceph RGW — IAM / STS / S3 Tenant Demo

This notebook demonstrates a complete multi-tenant identity and access management workflow
using the **Ceph RADOS Gateway (RGW)** IAM-compatible API.

| Step | What happens |
|------|--------------|
| **1** | Install libs and configure admin client |
| **2** | Platform admin creates a **tenant** with a **Tenant Administrator** user |
| **3** | Tenant Admin creates **subaccount** users within the tenant |
| **4** | Tenant Admin creates a **tenant-scoped IAM role** with an S3 permission policy |
| **5** | Spin up a local **OIDC / JWKS server** and register it with RGW |
| **6** | Generate a signed **ID token** (JWT) for the subaccount user |
| **7** | Subaccount calls **`AssumeRoleWithWebIdentity`** to obtain STS temporary credentials |
| **8** | Use STS credentials to perform **S3 bucket operations** |
| **9** | Clean up all created resources |

> **Prerequisites:** The Ceph cluster must be running (`docker compose up -d --wait`).  
> The notebook talks to RGW at the IP on `ceph-net`; adjust `RGW_ENDPOINT` below if you have
> host ports published.

In [1]:
# Install required packages
%pip install boto3 PyJWT cryptography requests -q
print("✓ Packages ready")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 1.44.51 requires botocore==1.42.61, but you have botocore 1.42.67 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
✓ Packages ready


## Section 1 — Configuration and Admin Client

Edit `RGW_ENDPOINT` and `JWKS_HOST` to match your environment.
- **`RGW_ENDPOINT`** — RGW container IP on `ceph-net` (or `http://localhost:7480` if ports are published)
- **`JWKS_HOST`** — IP reachable *from inside the RGW container* to the notebook host.  
  Docker's gateway on `ceph-net` (`172.20.0.1`) works when the notebook runs on the same host.

In [ ]:
import boto3
import json
import time
import threading
import base64
import requests
from datetime import datetime, timezone
from http.server import HTTPServer, BaseHTTPRequestHandler
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend
import jwt as pyjwt

# ── RGW / cluster ────────────────────────────────────────────────────────────
RGW_ENDPOINT = "http://172.20.0.13:7480"   # ceph-net IP; or http://localhost:7480
REGION       = "us-east-1"

# ── Platform admin credentials (from setup.sh / .env) ────────────────────────
ADMIN_ACCESS_KEY = "iamaccess123456"
ADMIN_SECRET_KEY = "iamsecret1234567"

# ── Tenant namespace ──────────────────────────────────────────────────────────
TENANT       = "acmecorp"
TENANT_ADMIN = f"{TENANT}-admin"
TENANT_USER  = f"{TENANT}-user1"
TENANT_ROLE  = f"{TENANT}-s3role"
TENANT_BUCKET= f"{TENANT}-demo-bucket"

# ── OIDC / JWKS mini-server (runs on the notebook host) ──────────────────────
JWKS_HOST = "172.20.0.1"   # Docker gateway — reachable from inside RGW container
JWKS_PORT = 8877            # Must not conflict with other services

# ── Helpers ───────────────────────────────────────────────────────────────────
def pp(label, obj):
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print('─'*60)
    print(json.dumps(obj, indent=2, default=str))

def make_iam(access, secret):
    return boto3.client(
        "iam",
        endpoint_url=RGW_ENDPOINT,
        region_name=REGION,
        aws_access_key_id=access,
        aws_secret_access_key=secret,
    )

def make_sts(access, secret):
    return boto3.client(
        "sts",
        endpoint_url=RGW_ENDPOINT,
        region_name=REGION,
        aws_access_key_id=access,
        aws_secret_access_key=secret,
    )

def make_s3(access, secret, token=None):
    return boto3.client(
        "s3",
        endpoint_url=RGW_ENDPOINT,
        region_name=REGION,
        aws_access_key_id=access,
        aws_secret_access_key=secret,
        aws_session_token=token,
    )

# Platform admin IAM client
iam_admin = make_iam(ADMIN_ACCESS_KEY, ADMIN_SECRET_KEY)

print("✓ Configuration loaded")
print(f"  RGW endpoint : {RGW_ENDPOINT}")
print(f"  Tenant       : {TENANT}")
print(f"  JWKS server  : http://{JWKS_HOST}:{JWKS_PORT}")

## Section 2 — Create Tenant and Tenant Administrator

The **platform admin** (`iamadmin`) creates:
1. A **Tenant Administrator** IAM user (`acmecorp-admin`)
2. An inline policy granting the Tenant Admin full IAM + STS rights
3. An access-key pair for the Tenant Admin so it can authenticate independently

> In Ceph RGW, tenancy is modelled through a shared user namespace.  
> Naming convention (`acmecorp-*`) keeps all tenant resources identifiable.

In [ ]:
# ── 2a. Create Tenant Administrator user ─────────────────────────────────────
try:
    resp = iam_admin.create_user(UserName=TENANT_ADMIN)
    pp("Created Tenant Admin", resp["User"])
except iam_admin.exceptions.EntityAlreadyExistsException:
    print(f"ℹ  User '{TENANT_ADMIN}' already exists — continuing")

# ── 2b. Attach inline admin policy (full IAM + STS + S3) ─────────────────────
TENANT_ADMIN_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "iam:*",
                "sts:*",
                "s3:*"
            ],
            "Resource": "*"
        }
    ]
})

iam_admin.put_user_policy(
    UserName=TENANT_ADMIN,
    PolicyName=f"{TENANT_ADMIN}-admin-policy",
    PolicyDocument=TENANT_ADMIN_POLICY,
)
print(f"✓ Admin policy attached to '{TENANT_ADMIN}'")

# ── 2c. Create or reuse access keys for Tenant Admin ─────────────────────────
existing_keys = iam_admin.list_access_keys(UserName=TENANT_ADMIN)["AccessKeyMetadata"]

if existing_keys:
    print(f"ℹ  '{TENANT_ADMIN}' already has {len(existing_keys)} key(s).")
    print("   Delete them first if you want fresh keys, or enter existing credentials:")
    TENANT_ADMIN_ACCESS = input(f"  AccessKeyId for '{TENANT_ADMIN}': ").strip()
    TENANT_ADMIN_SECRET = input(f"  SecretAccessKey for '{TENANT_ADMIN}': ").strip()
else:
    resp = iam_admin.create_access_key(UserName=TENANT_ADMIN)
    ak = resp["AccessKey"]
    TENANT_ADMIN_ACCESS = ak["AccessKeyId"]
    TENANT_ADMIN_SECRET = ak["SecretAccessKey"]
    print(f"✓ Access key created for '{TENANT_ADMIN}'")

print(f"\n  AccessKeyId : {TENANT_ADMIN_ACCESS}")
print(f"  SecretKey   : {TENANT_ADMIN_SECRET[:8]}{'*'*12}")

# Build Tenant Admin IAM client
iam_tenant = make_iam(TENANT_ADMIN_ACCESS, TENANT_ADMIN_SECRET)
print(f"\n✓ Tenant Admin IAM client ready")

## Section 3 — Tenant Administrator Creates Subaccounts

Acting as the **Tenant Admin**, we create a subaccount user (`acmecorp-user1`).  
The subaccount is granted only a minimal STS policy — it will obtain S3 access  
exclusively through **role assumption**, not through direct S3 credentials.

In [ ]:
# ── 3a. Create subaccount user (as Tenant Admin) ─────────────────────────────
try:
    resp = iam_tenant.create_user(UserName=TENANT_USER)
    pp("Created Subaccount User", resp["User"])
except iam_tenant.exceptions.EntityAlreadyExistsException:
    print(f"ℹ  User '{TENANT_USER}' already exists — continuing")

# ── 3b. Minimal policy: subaccount may only call STS ─────────────────────────
SUBUSER_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowSTSOnly",
            "Effect": "Allow",
            "Action": [
                "sts:AssumeRole",
                "sts:AssumeRoleWithWebIdentity",
            ],
            "Resource": "*"
        }
    ]
})

iam_tenant.put_user_policy(
    UserName=TENANT_USER,
    PolicyName=f"{TENANT_USER}-sts-only",
    PolicyDocument=SUBUSER_POLICY,
)
print(f"✓ STS-only policy attached to subaccount '{TENANT_USER}'")

# ── 3c. Create or reuse access keys for subaccount ───────────────────────────
existing_keys = iam_tenant.list_access_keys(UserName=TENANT_USER)["AccessKeyMetadata"]

if existing_keys:
    print(f"ℹ  '{TENANT_USER}' already has {len(existing_keys)} key(s).")
    SUBUSER_ACCESS = input(f"  AccessKeyId for '{TENANT_USER}': ").strip()
    SUBUSER_SECRET = input(f"  SecretAccessKey for '{TENANT_USER}': ").strip()
else:
    resp = iam_tenant.create_access_key(UserName=TENANT_USER)
    ak = resp["AccessKey"]
    SUBUSER_ACCESS = ak["AccessKeyId"]
    SUBUSER_SECRET = ak["SecretAccessKey"]
    print(f"✓ Access key created for subaccount '{TENANT_USER}'")

print(f"\n  AccessKeyId : {SUBUSER_ACCESS}")
print(f"  SecretKey   : {SUBUSER_SECRET[:8]}{'*'*12}")

# ── 3d. List all roles visible to the Tenant Admin ───────────────────────────
roles = iam_tenant.list_roles()["Roles"]
print(f"\n  Roles visible to Tenant Admin ({len(roles)}):")
for r in roles:
    print(f"    - {r['RoleName']}  ({r['Arn']})")

## Section 4 — Tenant-Specific IAM Role with Policy Documents

The **Tenant Admin** creates a role that:
- **Trust policy** — allows OIDC-authenticated identities (WebIdentity) whose `sub` claim
  matches the subaccount user to assume the role
- **Permission policy** — grants scoped S3 read/write access, restricted to buckets
  whose name starts with the tenant prefix (`acmecorp-*`)

The role ARN is captured for use in the STS call.

In [ ]:
OIDC_ISSUER = f"http://{JWKS_HOST}:{JWKS_PORT}"
OIDC_PRINCIPAL = f"{JWKS_HOST}:{JWKS_PORT}"   # host:port form used in ARN condition keys

# ── 4a. Trust policy — OIDC WebIdentity ──────────────────────────────────────
#   Principal is the OIDC provider ARN; Condition locks down to this subaccount user.
TRUST_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Federated": f"arn:aws:iam:::oidc-provider/{OIDC_PRINCIPAL}"
            },
            "Action": "sts:AssumeRoleWithWebIdentity",
            "Condition": {
                "StringEquals": {
                    f"{OIDC_PRINCIPAL}:sub": TENANT_USER,
                    f"{OIDC_PRINCIPAL}:aud": TENANT_USER,
                }
            }
        }
    ]
})

# ── 4b. Permission policy — scoped S3 access ─────────────────────────────────
S3_PERMISSION_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "TenantBucketAccess",
            "Effect": "Allow",
            "Action": [
                "s3:ListAllMyBuckets",
                "s3:CreateBucket",
                "s3:DeleteBucket",
                "s3:ListBucket",
                "s3:GetObject",
                "s3:PutObject",
                "s3:DeleteObject",
            ],
            "Resource": [
                f"arn:aws:s3:::{TENANT}-*",
                f"arn:aws:s3:::{TENANT}-*/*",
            ]
        }
    ]
})

pp("Trust Policy", json.loads(TRUST_POLICY))
pp("Permission Policy", json.loads(S3_PERMISSION_POLICY))

# ── 4c. Create the role ───────────────────────────────────────────────────────
try:
    resp = iam_tenant.create_role(
        RoleName=TENANT_ROLE,
        AssumeRolePolicyDocument=TRUST_POLICY,
        Description=f"Scoped S3 role for {TENANT} OIDC-authenticated users",
    )
    ROLE_ARN = resp["Role"]["Arn"]
    print(f"\n✓ Role created: {ROLE_ARN}")
except iam_tenant.exceptions.EntityAlreadyExistsException:
    roles = iam_tenant.list_roles()["Roles"]
    ROLE_ARN = next(r["Arn"] for r in roles if r["RoleName"] == TENANT_ROLE)
    print(f"ℹ  Role '{TENANT_ROLE}' already exists: {ROLE_ARN}")

# ── 4d. Attach permission policy to role ───────────────────────────────────────
iam_tenant.put_role_policy(
    RoleName=TENANT_ROLE,
    PolicyName=f"{TENANT_ROLE}-s3-permissions",
    PolicyDocument=S3_PERMISSION_POLICY,
)
print(f"✓ S3 permission policy attached to role '{TENANT_ROLE}'")
print(f"\n  Role ARN: {ROLE_ARN}")

## Section 5 — OIDC Provider: Key Generation, JWKS Server, and Registration

To issue real OIDC tokens, this notebook runs a **self-contained JWKS/OIDC discovery server**
in a background thread:

```
http://<JWKS_HOST>:<JWKS_PORT>/.well-known/openid-configuration  ← discovery doc
http://<JWKS_HOST>:<JWKS_PORT>/.well-known/jwks.json             ← public key set
```

Steps:
1. Generate an **RSA-2048 key pair** in memory
2. Expose the public key as a **JSON Web Key Set (JWKS)**
3. Start the HTTP server on `0.0.0.0:JWKS_PORT` (reachable from RGW via `JWKS_HOST`)
4. Register the provider URL with Ceph RGW via `CreateOpenIDConnectProvider`

In [ ]:
# ── 5a. Generate RSA-2048 key pair ───────────────────────────────────────────
_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
    backend=default_backend(),
)
_public_key  = _private_key.public_key()
_pub_numbers = _public_key.public_numbers()

PRIVATE_KEY_PEM = _private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.TraditionalOpenSSL,
    encryption_algorithm=serialization.NoEncryption(),
)

KID = "lab-oidc-key-1"

def _int_to_b64url(n: int) -> str:
    length = (n.bit_length() + 7) // 8
    return base64.urlsafe_b64encode(n.to_bytes(length, "big")).rstrip(b"=").decode()

JWK_SET = {
    "keys": [
        {
            "kty": "RSA",
            "use": "sig",
            "alg": "RS256",
            "kid": KID,
            "n":   _int_to_b64url(_pub_numbers.n),
            "e":   _int_to_b64url(_pub_numbers.e),
        }
    ]
}

OIDC_DISCOVERY = {
    "issuer":                                OIDC_ISSUER,
    "jwks_uri":                              f"{OIDC_ISSUER}/.well-known/jwks.json",
    "response_types_supported":              ["id_token"],
    "subject_types_supported":               ["public"],
    "id_token_signing_alg_values_supported": ["RS256"],
}

print("✓ RSA-2048 key pair generated")
print(f"  KID : {KID}")
print(f"  n   : {JWK_SET['keys'][0]['n'][:32]}…")

In [ ]:
# ── 5b. Start OIDC/JWKS HTTP server in background thread ─────────────────────
_jwks_json      = json.dumps(JWK_SET).encode()
_discovery_json = json.dumps(OIDC_DISCOVERY).encode()

class _OIDCHandler(BaseHTTPRequestHandler):
    _routes = {
        "/.well-known/openid-configuration": _discovery_json,
        "/.well-known/jwks.json":            _jwks_json,
    }

    def do_GET(self):
        body = self._routes.get(self.path)
        if body is None:
            self.send_response(404); self.end_headers(); return
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, fmt, *args):
        print(f"  [OIDC server] {self.address_string()} {fmt % args}")

# Stop any previously running server (safe on re-run)
try:
    _oidc_server.shutdown()
    print("ℹ  Previous OIDC server stopped")
except NameError:
    pass

_oidc_server = HTTPServer(("0.0.0.0", JWKS_PORT), _OIDCHandler)
threading.Thread(target=_oidc_server.serve_forever, daemon=True).start()
print(f"✓ OIDC server listening on 0.0.0.0:{JWKS_PORT}")

# Self-test from notebook host
r = requests.get(f"http://localhost:{JWKS_PORT}/.well-known/openid-configuration", timeout=5)
r.raise_for_status()
print(f"✓ Discovery endpoint OK — issuer: {r.json()['issuer']}")

r = requests.get(f"http://localhost:{JWKS_PORT}/.well-known/jwks.json", timeout=5)
r.raise_for_status()
print(f"✓ JWKS endpoint OK — {len(r.json()['keys'])} key(s) published")

In [ ]:
# ── 5c. Register OIDC provider with Ceph RGW ─────────────────────────────────
#
# ThumbprintList is required by the API but not validated by RGW for HTTP endpoints.
# A 40-character dummy is used here; in production use the real TLS cert SHA-1 fingerprint.
#
try:
    resp = iam_admin.create_open_id_connect_provider(
        Url=OIDC_ISSUER,
        ClientIDList=[TENANT_USER],          # audiences this provider issues tokens for
        ThumbprintList=["0" * 40],           # dummy — not checked for HTTP in Ceph
    )
    OIDC_ARN = resp["OpenIDConnectProviderArn"]
    print(f"✓ OIDC provider registered")
except iam_admin.exceptions.EntityAlreadyExistsException:
    OIDC_ARN = f"arn:aws:iam:::oidc-provider/{OIDC_PRINCIPAL}"
    print(f"ℹ  OIDC provider already registered")

print(f"  ARN: {OIDC_ARN}")

In [ ]:
# ── 5d. Generate signed OIDC ID Token (JWT) for the subaccount user ──────────
now = int(time.time())

id_token_claims = {
    "iss": OIDC_ISSUER,                # must match registered provider URL
    "sub": TENANT_USER,                # subject — must match role trust policy condition
    "aud": TENANT_USER,                # audience — must match ClientIDList above
    "iat": now,
    "exp": now + 3600,
    # Custom claims (informational)
    "name":     f"{TENANT} Subaccount User",
    "tenant":   TENANT,
    "email":    f"{TENANT_USER}@{TENANT}.example.com",
}

ID_TOKEN = pyjwt.encode(
    id_token_claims,
    PRIVATE_KEY_PEM,
    algorithm="RS256",
    headers={"kid": KID},
)

# Display decoded claims (no signature verification — just for inspection)
decoded = pyjwt.decode(ID_TOKEN, options={"verify_signature": False})
pp("ID Token Claims", decoded)
print(f"\n✓ ID Token signed with RS256 (kid={KID})")
print(f"  Token (first 60 chars): {ID_TOKEN[:60]}…")

## Section 6 — STS: AssumeRoleWithWebIdentity

The subaccount user now calls **`AssumeRoleWithWebIdentity`** passing:
- **`RoleArn`** — the tenant role ARN created in Section 4
- **`WebIdentityToken`** — the RS256-signed JWT from Section 5
- **`RoleSessionName`** — an identifying label for this session

RGW validates the JWT:
1. Fetches `<issuer>/.well-known/openid-configuration` → finds `jwks_uri`
2. Fetches `jwks_uri` → finds the public key matching `kid`
3. Verifies the JWT signature and checks `sub` / `aud` conditions in the trust policy
4. Issues temporary credentials (AccessKeyId, SecretAccessKey, SessionToken)

In [ ]:
# ── 6. AssumeRoleWithWebIdentity ─────────────────────────────────────────────
sts_client = make_sts(SUBUSER_ACCESS, SUBUSER_SECRET)

resp = sts_client.assume_role_with_web_identity(
    RoleArn=ROLE_ARN,
    RoleSessionName=f"{TENANT_USER}-oidc-session",
    WebIdentityToken=ID_TOKEN,
    DurationSeconds=3600,
)

creds       = resp["Credentials"]
STS_ACCESS  = creds["AccessKeyId"]
STS_SECRET  = creds["SecretAccessKey"]
STS_TOKEN   = creds["SessionToken"]
STS_EXPIRY  = creds["Expiration"]

assumed_user = resp.get("AssumedRoleUser", {})

pp("AssumeRoleWithWebIdentity response", {
    "AssumedRoleUser": assumed_user,
    "Credentials": {
        "AccessKeyId":    STS_ACCESS,
        "SecretKey":      STS_SECRET[:8] + "…",
        "SessionToken":   STS_TOKEN[:20] + "…",
        "Expiration":     STS_EXPIRY,
    },
    "Provider": resp.get("Provider"),
    "Audience":  resp.get("Audience"),
})

print(f"\n✓ Temporary credentials obtained!")
print(f"  Expires: {STS_EXPIRY}")

## Section 7 — S3 Bucket Operations Using STS Credentials

The **temporary credentials** from the STS call are now used to drive the full S3 lifecycle:

| Operation | boto3 call |
|---|---|
| Create bucket | `create_bucket` |
| List buckets | `list_buckets` |
| Upload object | `put_object` |
| List objects | `list_objects_v2` |
| Download and verify | `get_object` |
| Delete object | `delete_object` |
| Delete bucket | `delete_bucket` |

The bucket name starts with `acmecorp-`, so it falls within the permission policy's `Resource` scope.

In [ ]:
# ── 7. S3 operations with STS credentials ────────────────────────────────────
s3 = make_s3(STS_ACCESS, STS_SECRET, STS_TOKEN)

OBJECT_KEY  = "demo/hello.txt"
OBJECT_BODY = (
    f"Written by : {TENANT_USER}\n"
    f"Role       : {TENANT_ROLE}\n"
    f"Timestamp  : {datetime.now(timezone.utc).isoformat()}\n"
    f"Auth       : OIDC → STS AssumeRoleWithWebIdentity\n"
)

# ── 7a. Create bucket ─────────────────────────────────────────────────────────
s3.create_bucket(Bucket=TENANT_BUCKET)
print(f"✓ Bucket created : s3://{TENANT_BUCKET}")

# ── 7b. List all buckets (role sees only tenant-scoped buckets) ───────────────
buckets = s3.list_buckets().get("Buckets", [])
print(f"\n✓ Buckets ({len(buckets)}):")
for b in buckets:
    print(f"    {b['Name']}  (created {b['CreationDate'].strftime('%Y-%m-%d %H:%M:%S UTC')})")

# ── 7c. Upload object ─────────────────────────────────────────────────────────
s3.put_object(Bucket=TENANT_BUCKET, Key=OBJECT_KEY, Body=OBJECT_BODY.encode())
print(f"\n✓ Uploaded  : s3://{TENANT_BUCKET}/{OBJECT_KEY}")

# ── 7d. List objects ──────────────────────────────────────────────────────────
objs = s3.list_objects_v2(Bucket=TENANT_BUCKET).get("Contents", [])
print(f"\n✓ Objects in bucket ({len(objs)}):")
for o in objs:
    print(f"    {o['Key']}  ({o['Size']} bytes,  ETag: {o['ETag']})")

# ── 7e. Download and verify ───────────────────────────────────────────────────
downloaded = s3.get_object(Bucket=TENANT_BUCKET, Key=OBJECT_KEY)["Body"].read().decode()
print(f"\n✓ Downloaded object content:")
print("  " + downloaded.replace("\n", "\n  ").rstrip())
assert downloaded == OBJECT_BODY, "Content mismatch!"
print("  ✓ Round-trip content matches")

## Section 8 — Cleanup

Remove all resources created during this notebook run:
- S3 objects and bucket
- IAM role and its inline policy
- Subaccount user, access key, inline policy
- Tenant Admin user, access key, inline policy
- OIDC provider registration
- Stop the background JWKS server

In [ ]:
def _safe(label, fn):
    try:
        fn()
        print(f"  ✓ {label}")
    except Exception as exc:
        print(f"  ⚠ {label}: {exc}")

print("── S3 cleanup ───────────────────────────────────────────────────────")
_safe("delete object",  lambda: s3.delete_object(Bucket=TENANT_BUCKET, Key=OBJECT_KEY))
_safe("delete bucket",  lambda: s3.delete_bucket(Bucket=TENANT_BUCKET))

print("\n── IAM role cleanup (as Tenant Admin) ───────────────────────────────")
_safe("delete role inline policy",
      lambda: iam_tenant.delete_role_policy(
          RoleName=TENANT_ROLE, PolicyName=f"{TENANT_ROLE}-s3-permissions"))
_safe("delete role",
      lambda: iam_tenant.delete_role(RoleName=TENANT_ROLE))

print("\n── Subaccount cleanup (as Tenant Admin) ─────────────────────────────")
for ak in iam_tenant.list_access_keys(UserName=TENANT_USER)["AccessKeyMetadata"]:
    _safe(f"delete access key {ak['AccessKeyId']}",
          lambda k=ak["AccessKeyId"]: iam_tenant.delete_access_key(
              UserName=TENANT_USER, AccessKeyId=k))
_safe("delete subaccount inline policy",
      lambda: iam_tenant.delete_user_policy(
          UserName=TENANT_USER, PolicyName=f"{TENANT_USER}-sts-only"))
_safe("delete subaccount user",
      lambda: iam_tenant.delete_user(UserName=TENANT_USER))

print("\n── Tenant Admin cleanup (as platform admin) ─────────────────────────")
for ak in iam_admin.list_access_keys(UserName=TENANT_ADMIN)["AccessKeyMetadata"]:
    _safe(f"delete access key {ak['AccessKeyId']}",
          lambda k=ak["AccessKeyId"]: iam_admin.delete_access_key(
              UserName=TENANT_ADMIN, AccessKeyId=k))
_safe("delete tenant-admin inline policy",
      lambda: iam_admin.delete_user_policy(
          UserName=TENANT_ADMIN, PolicyName=f"{TENANT_ADMIN}-admin-policy"))
_safe("delete tenant-admin user",
      lambda: iam_admin.delete_user(UserName=TENANT_ADMIN))

print("\n── OIDC provider cleanup ────────────────────────────────────────────")
_safe("delete OIDC provider",
      lambda: iam_admin.delete_open_id_connect_provider(
          OpenIDConnectProviderArn=OIDC_ARN))

print("\n── JWKS server shutdown ─────────────────────────────────────────────")
_safe("stop JWKS server", lambda: _oidc_server.shutdown())

print("\n✓ Cleanup complete")

## Summary

This notebook demonstrated the complete **IAM → OIDC → STS → S3** workflow on Ceph RGW:

```
Platform Admin (iamadmin)
  └─ creates → Tenant Admin (acmecorp-admin)
                  └─ creates → Subaccount (acmecorp-user1)
                  └─ creates → IAM Role (acmecorp-s3role)
                                  ├─ trust policy  : OIDC issuer + sub==acmecorp-user1
                                  └─ perm policy   : s3:* on arn:aws:s3:::acmecorp-*

OIDC Flow (self-signed, local JWKS server):
  Notebook JWKS server  ──JWKS──►  Ceph RGW validates JWT
  acmecorp-user1 (JWT)  ──AssumeRoleWithWebIdentity──►  STS temp creds

S3 (temp creds):
  create bucket → upload object → list → download → delete → delete bucket
```

### Key takeaways

| Topic | Detail |
|---|---|
| **Tenancy model** | Naming convention (`acmecorp-*`) scopes users, roles, and buckets |
| **OIDC validation** | RGW fetches JWKS from `<issuer>/.well-known/jwks.json` and verifies RS256 JWT |
| **Credential lifetime** | STS tokens expire (here: 1 hour); refresh by calling `AssumeRoleWithWebIdentity` again |
| **Policy scoping** | Permission policy restricts S3 access to `arn:aws:s3:::acmecorp-*` |
| **`rgw_sts_key`** | Must be exactly 16 bytes; pass via `--rgw-sts-key` CLI flag, **not** via config DB |